In [ ]:
class KnowledgeDistillation(nn.Module):
    """
    Knowledge distillation loss for model compression
    Temperature parameter controls softness of probability distributions
    """
    def __init__(self, temperature=4.0, alpha=0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_loss = nn.KLDivLoss(reduction='batchmean')
    
    def forward(self, student_output, teacher_output, target):
        # Hard target loss (CE)
        ce = self.ce_loss(student_output, target)
        
        # Soft target loss (KL divergence)
        student_soft = F.log_softmax(student_output / self.temperature, dim=1)
        teacher_soft = F.softmax(teacher_output / self.temperature, dim=1)
        kl = self.kl_loss(student_soft, teacher_soft)
        
        # Combined loss
        loss = self.alpha * ce + (1 - self.alpha) * kl
        return loss

kd_loss = KnowledgeDistillation(temperature=4.0, alpha=0.7)
print("Knowledge distillation loss created")

## 2. Knowledge Distillation for Model Compression

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MultiHeadAttention(nn.Module):
    """
    Multi-head attention mechanism
    """
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.heads = heads
        self.dim = dim
        self.scale = (dim // heads) ** -0.5
        
        self.to_qkv = nn.Linear(dim, dim * 3)
        self.to_out = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        b, n, _ = x.shape
        qkv = self.to_qkv(x).reshape(b, n, 3, self.heads, self.dim // self.heads)
        q, k, v = map(lambda t: t.transpose(1, 3), qkv.chunk(3, dim=2))
        
        dots = torch.einsum('bhid,bhjd->bhij', q, k) * self.scale
        attn = dots.softmax(dim=-1)
        
        out = torch.einsum('bhij,bhjd->bhid', attn, v)
        out = out.transpose(1, 3).reshape(b, n, -1)
        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                MultiHeadAttention(dim, heads, dropout),
                nn.Sequential(
                    nn.LayerNorm(dim),
                    nn.Linear(dim, mlp_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(mlp_dim, dim),
                    nn.Dropout(dropout)
                )
            ]))
    
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return x

class ViT(nn.Module):
    def __init__(self, image_size, patch_size, num_classes, dim, depth, heads, mlp_dim, channels=3):
        super().__init__()
        assert image_size % patch_size == 0
        
        num_patches = (image_size // patch_size) ** 2
        patch_dim = channels * patch_size * patch_size
        
        self.patch_embedding = nn.Linear(patch_dim, dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        
        self.transformer = Transformer(dim, depth, heads, mlp_dim)
        self.to_latent = nn.Identity()
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )
    
    def forward(self, img):
        b, c, h, w = img.shape
        p = 16  # patch size
        
        x = img.reshape(b, c, h // p, p, w // p, p)
        x = x.permute(0, 2, 4, 1, 3, 5).reshape(b, (h // p) * (w // p), -1)
        
        x = self.patch_embedding(x)
        cls = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embedding
        
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.to_latent(x)
        return self.mlp_head(x)

# Create ViT model
vit = ViT(
    image_size=224,
    patch_size=16,
    num_classes=1000,
    dim=768,
    depth=12,
    heads=12,
    mlp_dim=3072
).to(device)

print("Vision Transformer created successfully")
print(f"Parameters: {sum(p.numel() for p in vit.parameters()):,}")

## 1. Vision Transformer (ViT)

# Advanced Topics and Multi-Modal Learning
## Combining Vision with Other Modalities

Topics covered:
- Vision Transformer (ViT)
- CLIP (Contrastive Language-Image Pre-training)
- Self-supervised learning (SimCLR, MoCo)
- Vision-language models
- Knowledge distillation and model compression
- Adversarial robustness
- Explainability and interpretability
- Few-shot and zero-shot learning